# Aruba population trends, 2015-2023


Source file: Table-1.12-Departures-by-country-of-birth-and-sex.xlsx\
Source: CBS Aruba and the Population Registry Office

---
## 1. Setup

## Imports and paths

This block of code is necessary to create a reproducible notebook environment.

In [65]:
# Importing necessary libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config.project_paths import (
    ROOT,
    DATA_RAW,
    FIGURES
)

In [66]:
# Verify all paths to ensure stable environment

print("ROOT:", ROOT)
print("RAW DATA:", DATA_RAW)
print("FIGURES:", FIGURES)

ROOT: /home/ggirelli/Documents/DataAnalysis/projects/cbs_aruba
RAW DATA: /home/ggirelli/Documents/DataAnalysis/projects/cbs_aruba/data/raw
FIGURES: /home/ggirelli/Documents/DataAnalysis/projects/cbs_aruba/outputs/figures


In [67]:
DEPARTURES_COUNTRY_SEX = DATA_RAW / "Table-1.12-Departures-by-country-of-birth-and-sex.xlsx"

In [68]:
# # Stop early if the source file is missing

if not DEPARTURES_COUNTRY_SEX.exists():
    raise FileNotFoundError

## 2. Load and inspect source table

In [69]:
# Read raw Excel file

raw_df = pd.read_excel(DEPARTURES_COUNTRY_SEX, skiprows=1, header=None)

In [70]:
# Drop empty row 3 in the data frame and turn all year values into Int64

df = raw_df.dropna(how="all")
df.iloc[0] = pd.to_numeric(df.iloc[0], errors="coerce").astype("Int64")

In [71]:
# Promote first row to columns and reset index

df.columns = df.iloc[0]
df = df.iloc[1:].reset_index(drop=True)

# Turn columns to a list type ad name the empyt cell at index 0 'indicator'

cols = df.columns.tolist()
cols[0] = "indicator"
df.columns = cols

cols = pd.Series(df.columns)
df.columns = cols.ffill()

In [72]:
clean_df = df

### Building a new list of column names using a for loop

The next block of code builds a new list of column names from two header layers by combining them
- the top row (years)
- and the bottom row (Male, Female)

In [73]:
header_top = pd.Series(clean_df.columns)
header_bottom = clean_df.loc[0]

# for top, bottom in zip(header_top, header_bottom):
#     print("top   =", top)
#     print("bottom=", bottom)
#     print("---")

# Create an an empty list object where new column names will be stored
new_cols = []

# Loop that pairs values together
for top, bottom in zip(header_top, header_bottom):
    if pd.isna(top):                                  # If top header is missing, use only bottom label
        new_cols.append(str(bottom).strip())
    elif str(top) == "Indicator":
        new_cols.append("Indicator")
    else:
        new_cols.append(f"{top}_{str(bottom).strip()}")

In [74]:
header_bottom.head(5)

indicator    Country
2015            Male
2015          Female
2016            Male
2016          Female
Name: 0, dtype: object

In [77]:
# Reset index

clean_df.columns = new_cols
clean_df = clean_df.drop(index=0).reset_index(drop=True)

clean_df

,indicator_Country,2015_Male,2015_Female,2016_Male,2016_Female,2017_Male,2017_Female,2018_Male,2018_Female,2019_Male,2019_Female,2020_Male,2020_Female,2021_Male,2021_Female,2022_Male,2022_Female,2023_Male,2023_Female
0,Aruba/ Neth. Ant.,649,607,665,667,868,829,780,764,839,801,868,933,848,850,839,840,814,743
1,The Netherlands,303,287,298,300,326,310,352,344,283,300,286,299,210,237,209,210,214,207
2,Colombia,89,120,94,171,125,166,92,155,112,166,142,253,63,101,102,177,83,167
3,Venezuela,54,57,68,43,51,49,48,42,74,85,63,51,54,44,51,72,59,53
4,Dominican Republic,37,51,45,61,43,120,32,81,47,90,57,116,26,51,45,74,41,79
5,Other,244,197,216,256,249,275,204,210,224,224,198,197,97,131,158,155,155,145
6,Source: Population Registry Office,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
